In [ ]:



import time
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


SEED = 42
EPOCHS = 6
BATCH_SIZE = 128
LEARNING_RATE = 0.001

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Urządzenie:", device)
print("Epoki:", EPOCHS)



transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_ds = torchvision.datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_ds = torchvision.datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_dl = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
    num_workers=0
)

print("\nLiczba próbek treningowych:", len(train_ds))
print("Liczba próbek testowych:", len(test_ds))



class CNN(nn.Module):

    def __init__(self, n_classes=10, pool="max"):
        super().__init__()

        self.pool_type = pool

        def block(cin, cout):

            layers = [
                nn.Conv2d(cin, cout, kernel_size=3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU()
            ]

            if pool == "max":
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

            elif pool == "avg":
                layers.append(nn.AvgPool2d(kernel_size=2, stride=2))

            elif pool == "stride":
                layers.extend([
                    nn.Conv2d(
                        cout,
                        cout,
                        kernel_size=3,
                        stride=2,
                        padding=1
                    ),
                    nn.BatchNorm2d(cout),
                    nn.ReLU()
                ])

            return nn.Sequential(*layers)


        self.features = nn.Sequential(
            block(1, 32),
            block(32, 64)
        )


        self.gap = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

    def feature_shape(self):
        with torch.no_grad():
            x = torch.zeros(1, 1, 28, 28)
            return tuple(self.features(x).shape[1:])




def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )




def evaluate(model, dataloader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for xb, yb in dataloader:

            xb = xb.to(device)
            yb = yb.to(device)

            predictions = model(xb).argmax(dim=1)

            correct += (predictions == yb).sum().item()
            total += yb.size(0)

    return correct / total



def train_model(model, epochs=EPOCHS):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    accuracy_history = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for xb, yb in train_dl:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            outputs = model(xb)

            loss = criterion(outputs, yb)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()


        test_acc = evaluate(model, test_dl)

        accuracy_history.append(test_acc)

        avg_loss = running_loss / len(train_dl)

        print(
            f"Epoka {epoch+1}/{epochs} | "
            f"Loss: {avg_loss:.4f} | "
            f"Test accuracy: {test_acc*100:.2f}%"
        )

    return model, accuracy_history



variants = ["max", "avg", "stride"]

results = {}

print("\n" + "="*65)
print("ROZPOCZĘCIE EKSPERYMENTU")
print("="*65)

for variant in variants:

    print("\n" + "-"*55)

    if variant == "max":
        name = "Max Pooling 2x2"

    elif variant == "avg":
        name = "Average Pooling 2x2"

    else:
        name = "Convolution Stride 2"

    print("WARIANT:", name)
    print("-"*55)


    torch.manual_seed(SEED)

    model = CNN(pool=variant)

    parameters = count_parameters(model)

    feature_size = model.feature_shape()

    print("Liczba parametrów:", f"{parameters:,}")
    print("Mapa cech przed GAP:", feature_size)

    start_time = time.time()

    model, accuracy_history = train_model(
        model,
        epochs=EPOCHS
    )

    elapsed = time.time() - start_time

    results[variant] = {
        "name": name,
        "accuracy": accuracy_history,
        "last_accuracy": accuracy_history[-1],
        "parameters": parameters,
        "feature_size": feature_size,
        "time": elapsed
    }

    print("Czas treningu:", round(elapsed, 2), "sekund")




plt.figure(figsize=(10, 6))

for variant in variants:

    history = results[variant]["accuracy"]

    plt.plot(
        range(1, EPOCHS + 1),
        [x * 100 for x in history],
        marker="o",
        linewidth=2,
        label=results[variant]["name"]
    )

plt.xlabel("Epoka")
plt.ylabel("Accuracy testowa [%]")
plt.title("Porównanie krzywych accuracy dla różnych metod redukcji")

plt.xticks(range(1, EPOCHS + 1))
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()




print("\n")
print("="*100)
print("TABELA PORÓWNAWCZA WARIANTÓW CNN")
print("="*100)

print(
    f"{'Wariant':25} | "
    f"{'Accuracy [%]':>13} | "
    f"{'Parametry':>12} | "
    f"{'Mapa cech [C,H,W]':>20}"
)

print("-"*100)

for variant in variants:

    r = results[variant]

    print(
        f"{r['name']:25} | "
        f"{r['last_accuracy']*100:12.2f}% | "
        f"{r['parameters']:12,} | "
        f"{str(r['feature_size']):>20}"
    )

print("="*100)




print("\nCZAS TRENINGU:")

for variant in variants:

    r = results[variant]

    print(
        f"{r['name']:25} : {r['time']:.2f} s"
    )




best_accuracy = max(
    results,
    key=lambda x: results[x]["last_accuracy"]
)

fewest_parameters = min(
    results,
    key=lambda x: results[x]["parameters"]
)

most_parameters = max(
    results,
    key=lambda x: results[x]["parameters"]
)

print("\n")
print("="*65)
print("WNIOSKI Z EKSPERYMENTU")
print("="*65)

print(
    "\nNajwyższą accuracy testową uzyskał:",
    results[best_accuracy]["name"],
    f"({results[best_accuracy]['last_accuracy']*100:.2f}%)"
)

print(
    "\nNajmniejszą liczbę parametrów posiada:",
    results[fewest_parameters]["name"],
    f"({results[fewest_parameters]['parameters']:,})"
)

print(
    "\nNajwiększą liczbę parametrów posiada:",
    results[most_parameters]["name"],
    f"({results[most_parameters]['parameters']:,})"
)

print("\nKOMPROMIS MIĘDZY JAKOŚCIĄ A LICZBĄ PARAMETRÓW:")

for variant in variants:

    r = results[variant]

    print(
        f"\n{r['name']}: "
        f"accuracy {r['last_accuracy']*100:.2f}%, "
        f"parametry {r['parameters']:,}."
    )

print("""
PODSUMOWANIE:

1. Max Pooling 2x2 wybiera największą wartość z okna,
   dzięki czemu zachowuje najsilniejsze cechy obrazu.

2. Average Pooling 2x2 oblicza średnią z okna,
   przez co wygładza mapy cech i może usuwać część szczegółów.

3. Convolution Stride 2 wykorzystuje uczony splot z krokiem 2.
   Pozwala sieci samodzielnie nauczyć się redukcji wymiarów,
   ale zwiększa liczbę parametrów.

4. Global Average Pooling zmniejsza każdą mapę cech
   do jednej wartości i ogranicza liczbę parametrów
   końcowej warstwy klasyfikacyjnej.

5. Najlepszy wariant należy oceniać zarówno pod względem
   accuracy testowej, jak i liczby parametrów.

Wyniki zależą od inicjalizacji wag, przebiegu treningu
oraz zastosowanej metody redukcji.
""")

print("="*65)
print("EKSPERYMENT ZAKOŃCZONY")
print("="*65)


Urządzenie: cpu
Epoki: 6


100%|██████████| 26.4M/26.4M [00:03<00:00, 8.53MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 135kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.56MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.1MB/s]



Liczba próbek treningowych: 60000
Liczba próbek testowych: 10000

ROZPOCZĘCIE EKSPERYMENTU

-------------------------------------------------------
WARIANT: Max Pooling 2x2
-------------------------------------------------------
Liczba parametrów: 19,658
Mapa cech przed GAP: (64, 7, 7)
